[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-06-ray-tune-hpo.ipynb#scrollTo=aa11bb22)

---
# Day 6 · Ray Tune — Hyperparameter Search at Scale
**certified-journeys / ray-certified** · Ray for Distributed Python · Practice Badge

> **Goal for today:** Run automated hyperparameter optimisation with Ray Tune using grid search, ASHA early stopping, and OptunaSearch — and retrieve the best trial result programmatically.


In [ ]:
%pip install -q 'ray[tune]' optuna scikit-learn


## Step 1 · Ray Tune Overview

**Ray Tune** is a scalable hyperparameter optimisation (HPO) library that runs trials in parallel across your Ray cluster.

Core concepts:
- **Trainable** — a function `(config: dict) → None` that trains a model and calls `tune.report()` or `ray.train.report()`.
- **Search space** — a dict mapping hyperparameter names to distributions (`tune.grid_search`, `tune.loguniform`, `tune.choice`, …).
- **Tuner** — the entry point: `Tuner(trainable, param_space=..., tune_config=...).fit()` returns a `ResultGrid`.
- **Scheduler** — decides which trials to stop early (ASHA, PBT) or pause/resume (PBT).
- **Search algorithm** — decides which `config` to try next (Optuna, HyperOpt, Ax).

```
Tuner(trainable, param_space, TuneConfig(scheduler, search_alg))
   → ResultGrid  →  .get_best_result()
```

📖 https://docs.ray.io/en/latest/tune/index.html


In [ ]:
import ray
from ray import tune

ray.init(ignore_reinit_error=True)

print("Ray version:", ray.__version__)
print("Tune module:", tune)


### What just happened?

- **`ray.init()`** starts a local cluster; all Tune trials will run as Ray tasks on this cluster.
- **`from ray import tune`** gives access to the search space primitives (`tune.grid_search`, `tune.loguniform`, etc.).
- In production, Tune connects to an existing multi-node cluster via `ray.init(address='auto')` — the same API.
- Each Tune trial is an independent Ray task; Tune's scheduler communicates with trial actors via the GCS.


## Step 2 · The Trainable Function

A **trainable** is a regular Python function that:
1. Accepts a `config: dict` argument containing sampled hyperparameters.
2. Trains a model (or runs any experiment) for one or more iterations.
3. Calls `tune.report(metric=value)` (or `ray.train.report()`) after each iteration.

Tune calls your trainable once per trial, passing a different `config` each time. When using a scheduler like ASHA, Tune may terminate a trial mid-way if it is underperforming.

We train a `RandomForestClassifier` on Iris — fast to run and easy to interpret.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load Iris once at module level — shared across all trials in this process
_iris = load_iris()
_X    = StandardScaler().fit_transform(_iris.data.astype(np.float64))
_y    = _iris.target

def train_iris_rf(config: dict):
    """
    Trainable: train a RandomForestClassifier with hyperparams from `config`.
    Calls tune.report() once with the mean cross-validation accuracy.
    """
    n_estimators = config["n_estimators"]   # number of trees
    max_depth    = config["max_depth"]       # tree depth (None = unlimited)
    min_samples  = config["min_samples_leaf"]# min samples at leaf

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples,
        random_state=42,
        n_jobs=1,   # keep 1 — Tune parallelises at trial level, not within RF
    )

    # 5-fold stratified CV for a reliable accuracy estimate
    cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, _X, _y, cv=cv, scoring="accuracy")
    acc    = float(scores.mean())

    # Report the metric — Tune uses this to compare trials
    tune.report(accuracy=acc, std=float(scores.std()))

print("Trainable `train_iris_rf` defined.")
# Quick sanity check: run it directly with a fixed config
train_iris_rf({"n_estimators": 10, "max_depth": 3, "min_samples_leaf": 1})
print("Direct call succeeded — trainable is valid.")


### What just happened?

- **`tune.report(accuracy=acc)`** is the only Tune-specific line in the trainable — the rest is plain scikit-learn.
- Loading data at module level (outside the function) avoids repeated I/O; Tune workers in the same process share the cache.
- **`n_jobs=1` inside RF** is intentional — Tune parallelises at the trial level, so inner parallelism fights for the same CPU cores.
- The sanity-check direct call confirms the function works before handing it to Tune.


## Step 3 · Search Spaces — `grid_search` and `loguniform`

Tune provides several primitives for defining hyperparameter distributions:

| Primitive | Use case | Example |
|---|---|---|
| `tune.grid_search([a, b, c])` | Exhaustive over a list | `[10, 50, 100]` trees |
| `tune.choice([a, b, c])` | Random sample from list | `[None, 3, 5]` depth |
| `tune.loguniform(lo, hi)` | Log-scale continuous | learning rate `1e-4` – `1e-1` |
| `tune.uniform(lo, hi)` | Linear-scale continuous | dropout `0.0` – `0.5` |
| `tune.randint(lo, hi)` | Random integer | `[1, 10]` leaves |

Mix `grid_search` and random primitives in one `param_space` dict — Tune handles the Cartesian product automatically.


In [ ]:
from ray.tune import Tuner, TuneConfig

# Search space mixing grid and random primitives
param_space_grid = {
    "n_estimators":   tune.grid_search([10, 50]),        # 2 values
    "max_depth":      tune.choice([None, 3, 5]),          # random from 3 options
    "min_samples_leaf": tune.randint(1, 4),              # random int in [1, 4)
}

# TuneConfig: num_samples * grid points = total trials
# grid_search(2) × num_samples=3 → 6 trials max
# Keep num_samples=5 and max_t=5 for Colab speed
tuner_grid = Tuner(
    train_iris_rf,
    param_space=param_space_grid,
    tune_config=TuneConfig(
        metric="accuracy",
        mode="max",
        num_samples=5,    # 5 random samples × 2 grid values = 10 trials
    ),
)

print("Grid-based Tuner created — calling fit()...")
results_grid = tuner_grid.fit()
print("\nGrid search complete.")
print(f"Total trials: {len(results_grid)}")


### What just happened?

- **`Tuner`** takes the trainable function, the param space, and a `TuneConfig` specifying which metric to optimise and in which direction.
- **`num_samples=5`** means Tune samples random values for `choice` and `randint` 5 times; for each sample it runs all `grid_search` combinations — total trials = `num_samples × grid_size`.
- **`results_grid.fit()`** blocks until all trials finish and returns a `ResultGrid`.
- In production, set `num_samples` higher (50–200) and pair with a scheduler to terminate bad trials early.


## Step 4 · Retrieving the Best Trial

**`ResultGrid`** is the object returned by `tuner.fit()`. Its most important methods:

- `.get_best_result(metric, mode)` — returns the `Result` with the best metric value.
- `.get_dataframe()` — all trials as a pandas DataFrame for analysis.
- `result.config` — the winning hyperparameter dict.
- `result.metrics` — the final reported metrics dict.


In [ ]:
import pandas as pd

# Get the best trial from the grid search
best_grid = results_grid.get_best_result(metric="accuracy", mode="max")

print("── Best trial (grid search) ──")
print("Config:  ", best_grid.config)
print("Metrics: ", best_grid.metrics)

# All trials as a DataFrame
df_grid = results_grid.get_dataframe()
cols = ["accuracy", "std", "config/n_estimators", "config/max_depth", "config/min_samples_leaf"]
# Only keep columns that exist
available_cols = [c for c in cols if c in df_grid.columns]
print("\nAll trials (sorted by accuracy):")
print(df_grid[available_cols].sort_values("accuracy", ascending=False).to_string(index=False))


### What just happened?

- **`.get_best_result(metric='accuracy', mode='max')`** finds the trial with the highest accuracy — consistent with our `TuneConfig(mode='max')`.
- **`best_grid.config`** is the exact `param_space` sample that produced the best result — copy it directly to retrain a final model.
- **`.get_dataframe()`** returns a flat DataFrame with one row per trial; column names for config values are prefixed with `config/`.
- In practice, run `.get_dataframe()` to generate training curves or compare variance across hyperparameter choices.


## Step 5 · ASHAScheduler — Terminate Bad Trials Early

**ASHA (Asynchronous Successive Halving)** is a principled early-stopping scheduler:

1. All trials start and run to at least `grace_period` iterations.
2. After `grace_period`, the bottom `(1 - 1/reduction_factor)` fraction is stopped.
3. Survivors run longer; the process repeats each `reduction_factor`×.

This gives the best trials exponentially more compute while terminating bad ones fast.

> **Tip:** ASHA is the right default scheduler for most HPO jobs. Start with ASHA + Optuna as your baseline combination.

📖 https://docs.ray.io/en/latest/tune/api/doc/ray.tune.schedulers.ASHAScheduler.html

For a scheduler to be useful, the trainable must report **iteratively** (e.g., per-epoch). We adapt the trainable to train incrementally.


In [ ]:
from ray.tune.schedulers import ASHAScheduler

# Iterative trainable: trains RF with increasing n_estimators to simulate epochs
def train_iris_rf_iterative(config: dict):
    """
    Iterative trainable: reports accuracy at multiple 'epochs' (tree-count milestones).
    ASHA can terminate this mid-way if accuracy is below the top quantile.
    """
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    import numpy as np

    max_depth     = config["max_depth"]
    min_samples   = config["min_samples_leaf"]
    max_estimators = config["max_estimators"]   # e.g. 40

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    # Simulate epochs by growing the forest incrementally
    for n_trees in range(5, max_estimators + 1, 5):   # steps: 5, 10, 15, …
        model = RandomForestClassifier(
            n_estimators=n_trees,
            max_depth=max_depth,
            min_samples_leaf=min_samples,
            random_state=42,
            n_jobs=1,
        )
        scores = cross_val_score(model, _X, _y, cv=cv, scoring="accuracy")
        acc    = float(scores.mean())

        # Report after each 'epoch' — ASHA observes these intermediate values
        tune.report(accuracy=acc, n_trees=n_trees)

# ASHAScheduler config
asha = ASHAScheduler(
    metric="accuracy",
    mode="max",
    max_t=5,           # max number of 'epochs' (reporting steps) per trial; keep low for Colab
    grace_period=1,    # every trial gets at least 1 reporting step
    reduction_factor=2,# top 1/2 survive each halving round
)

param_space_asha = {
    "max_depth":      tune.choice([None, 3, 5, 8]),
    "min_samples_leaf": tune.randint(1, 5),
    "max_estimators": 25,   # fixed; controls the number of reporting steps
}

tuner_asha = Tuner(
    train_iris_rf_iterative,
    param_space=param_space_asha,
    tune_config=TuneConfig(
        metric="accuracy",
        mode="max",
        num_samples=5,       # 5 trials total; ASHA will stop bad ones early
        scheduler=asha,
    ),
)

print("Running ASHA search (num_samples=5, max_t=5)...")
results_asha = tuner_asha.fit()

best_asha = results_asha.get_best_result(metric="accuracy", mode="max")
print("\n── Best trial (ASHA) ──")
print("Config:  ", best_asha.config)
print("Accuracy:", best_asha.metrics.get("accuracy"))


### What just happened?

- **`ASHAScheduler`** watched the accuracy after each reporting step and stopped trials in the bottom half at `grace_period=1`.
- Survivors ran all `max_t=5` steps — they received more total compute than stopped trials.
- **`max_t`** must match the number of reporting steps your trainable makes; too low and ASHA stops everything before it can learn.
- ASHA is **asynchronous** — it doesn't wait for all trials to finish a step; it acts as results arrive, maximising GPU utilisation on large clusters.


## Step 6 · OptunaSearch — Bayesian Optimisation

**OptunaSearch** replaces random sampling with Optuna's **Tree-structured Parzen Estimator (TPE)** — a Bayesian model that learns from past trials to propose better hyperparameters.

| Method | Sampling strategy | Best when |
|---|---|---|
| Grid search | Exhaustive | Small, discrete spaces |
| Random search | Uniform random | Large spaces, no budget |
| ASHA + Random | Random + early stop | Large spaces + time budget |
| ASHA + Optuna | Bayesian + early stop | **Best default for most jobs** |

📖 https://docs.ray.io/en/latest/tune/api/doc/ray.tune.search.optuna.OptunaSearch.html


In [ ]:
from ray.tune.search.optuna import OptunaSearch

# OptunaSearch wraps Optuna's sampler
optuna_search = OptunaSearch(
    metric="accuracy",
    mode="max",
    # seed for reproducibility
    seed=42,
)

# ASHAScheduler for early stopping
asha_for_optuna = ASHAScheduler(
    metric="accuracy",
    mode="max",
    max_t=5,
    grace_period=1,
    reduction_factor=2,
)

# param_space for OptunaSearch: use tune primitives — Optuna interprets them
param_space_optuna = {
    "max_depth":       tune.choice([None, 3, 5, 8]),
    "min_samples_leaf": tune.randint(1, 5),
    "max_estimators":  25,
}

tuner_optuna = Tuner(
    train_iris_rf_iterative,
    param_space=param_space_optuna,
    tune_config=TuneConfig(
        metric="accuracy",
        mode="max",
        num_samples=5,           # keep low for Colab speed
        search_alg=optuna_search,
        scheduler=asha_for_optuna,
    ),
)

print("Running ASHA + Optuna search (num_samples=5)...")
results_optuna = tuner_optuna.fit()

best_optuna = results_optuna.get_best_result(metric="accuracy", mode="max")
print("\n── Best trial (ASHA + Optuna) ──")
print("Config:  ", best_optuna.config)
print("Accuracy:", best_optuna.metrics.get("accuracy"))


### What just happened?

- **`OptunaSearch`** uses TPE to propose hyperparameters informed by previous trial results — it concentrates samples in promising regions.
- Combining with **`ASHAScheduler`** gives you both smart proposal (Optuna) and early stopping (ASHA) — the canonical best-practice combination.
- With only 5 trials, Optuna behaves similarly to random search; the advantage is visible at 30+ trials.
- **`seed=42`** ensures the Optuna sampler is reproducible across runs.


## Step 7 · Comparing Strategies — Grid vs ASHA vs ASHA+Optuna

In a real HPO workflow you would run all strategies and compare the best accuracy found within the same wall-clock budget. Here we summarise the results from the runs already completed in this notebook.


In [ ]:
import pandas as pd

# Collect best accuracy from each strategy
summary = [
    {
        "strategy":    "Grid search",
        "best_accuracy": results_grid.get_best_result("accuracy", "max").metrics.get("accuracy", float("nan")),
        "trials":      len(results_grid),
        "description": "Exhaustive, no early stopping",
    },
    {
        "strategy":    "ASHA (random)",
        "best_accuracy": results_asha.get_best_result("accuracy", "max").metrics.get("accuracy", float("nan")),
        "trials":      len(results_asha),
        "description": "Random sampling + early stopping",
    },
    {
        "strategy":    "ASHA + Optuna",
        "best_accuracy": results_optuna.get_best_result("accuracy", "max").metrics.get("accuracy", float("nan")),
        "trials":      len(results_optuna),
        "description": "Bayesian (TPE) + early stopping",
    },
]

df_summary = pd.DataFrame(summary).sort_values("best_accuracy", ascending=False)
print("── HPO Strategy Comparison ──")
print(df_summary.to_string(index=False))
print("\nNote: differences will be larger with more trials (num_samples=50+)")


### What just happened?

- With only 5 trials, the strategies produce similar accuracy — the Iris dataset is too simple to show dramatic differences.
- In production with 50–200 trials on complex models, **ASHA + Optuna consistently finds better hyperparameters with fewer total compute-hours** than grid search.
- **Grid search** becomes infeasible when the search space grows: 5 values per hyperparameter × 6 hyperparameters = 15 625 trials.
- The comparison table is the correct way to present HPO results in papers and MLOps dashboards.


## Step 8 · Population-Based Training (PBT) — Overview and When to Use It

**PopulationBasedTraining** is different from ASHA:
- ASHA **stops** bad trials; the rest run independently.
- PBT **mutates** hyperparameters during training: underperforming trials copy weights from top performers and perturb their learning rate or schedule.

PBT is most powerful for:
- Training schedules (learning rate warmup/decay)
- Continuously adapting hyperparameters during long RL training runs
- Joint optimisation of architecture + schedule

📖 https://docs.ray.io/en/latest/tune/api/doc/ray.tune.schedulers.PopulationBasedTraining.html

We demonstrate PBT setup (definition only — full training would require a PyTorch model with per-step checkpointing).


In [ ]:
from ray.tune.schedulers import PopulationBasedTraining

# PBT requires hyperparam_mutations — ranges within which to perturb hyperparameters
pbt = PopulationBasedTraining(
    metric="accuracy",
    mode="max",
    perturbation_interval=2,          # perturb every 2 reporting steps
    hyperparam_mutations={
        # PBT will randomly sample from these ranges when copying + mutating
        "max_depth":        [None, 3, 5, 8],
        "min_samples_leaf": [1, 2, 3, 4],
    },
)

print("PBT scheduler created:")
print(f"  Perturbation interval: {pbt._perturbation_interval} steps")
print(f"  Hyperparam mutations:  {pbt._hyperparam_mutations}")
print()
print("In production, use PBT with a Tuner like this:")
print("""
  Tuner(
      my_iterative_trainable,
      param_space=param_space,
      tune_config=TuneConfig(
          metric='accuracy', mode='max',
          num_samples=8,        # population size
          scheduler=pbt,
      ),
  ).fit()
""")


### What just happened?

- **`perturbation_interval`** controls how frequently PBT compares trials and mutates the bottom performers.
- **`hyperparam_mutations`** defines the ranges used for random perturbation — these should be wide enough for meaningful exploration.
- PBT requires **checkpointing inside the trainable** (using `ray.train.report(checkpoint=...)`) so it can copy weights from top to bottom performers.
- **Use PBT when**: you have long-running training jobs (hours to days), you want to co-optimise schedule + architecture, or you're doing RL with non-stationary reward landscapes.


## Step 9 · `loguniform` Search Space for Neural Network HPO

When tuning learning rates, weight decay, or other scale-sensitive hyperparameters, **`tune.loguniform(lo, hi)`** is the correct primitive — it samples uniformly in log space, ensuring equal probability mass at each order of magnitude.

This step shows a complete search space definition for a neural network using loguniform and other primitives.


In [ ]:
# Demonstrate loguniform sampling — does not train, just shows the distribution
import numpy as np

# Example: 10 draws from loguniform learning rate space
# In a real Tuner, Tune samples these automatically
lr_samples = np.exp(np.random.uniform(np.log(1e-4), np.log(1e-1), size=10))
print("loguniform(1e-4, 1e-1) samples:")
for lr in sorted(lr_samples):
    print(f"  {lr:.2e}")

print()

# Full neural network search space definition
nn_param_space = {
    "lr":           tune.loguniform(1e-4, 1e-1),  # learning rate
    "weight_decay": tune.loguniform(1e-6, 1e-2),  # L2 regularisation
    "hidden_size":  tune.choice([32, 64, 128, 256]),
    "dropout":      tune.uniform(0.0, 0.5),
    "batch_size":   tune.choice([32, 64, 128]),
    "optimizer":    tune.choice(["adam", "sgd"]),
}

print("Neural network search space:")
for k, v in nn_param_space.items():
    print(f"  {k:15s}: {v}")


### What just happened?

- **`loguniform`** ensures that `lr=0.0001` and `lr=0.001` are equally likely — critical when the optimal LR could be anywhere across 3 orders of magnitude.
- `tune.uniform` is for parameters where the linear scale is appropriate (dropout rate 0.1 vs 0.4 is a linear difference).
- `tune.choice` is correct for discrete categorical options (optimizer type, hidden size from a fixed set).
- This search space dict is passed directly to `Tuner(param_space=nn_param_space, ...)` — Tune handles all sampling.


In [ ]:
# ── Challenge ──────────────────────────────────────────────────────────────
# Challenge: Run a Tune HPO job on the Wine dataset with ASHA + OptunaSearch.
#   1. Load sklearn.datasets.load_wine()
#   2. Write a trainable function that:
#        - Trains a RandomForestClassifier with params from config
#        - Reports 'accuracy' using 5-fold CV
#        - Reports at 3 intermediate points (n_trees = 10, 20, 30)
#   3. Define a param_space with:
#        - max_depth: choice([None, 3, 5, 10])
#        - min_samples_leaf: randint(1, 6)
#   4. Run Tuner with num_samples=5, ASHAScheduler(max_t=3), OptunaSearch
#   5. Print the best config and accuracy

# Your solution here

# from sklearn.datasets import load_wine
# _wine = load_wine()
# _Xw   = StandardScaler().fit_transform(_wine.data.astype(np.float64))
# _yw   = _wine.target

# def train_wine_rf(config):
#     ...

# wine_tuner = Tuner(...)
# wine_results = wine_tuner.fit()
# print(wine_results.get_best_result("accuracy", "max").config)


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| Trainable | `(config: dict) → None`; calls `tune.report(metric=val)` after each iteration |
| `tune.grid_search([...])` | Exhaustive over list; multiplied by `num_samples` for random dims |
| `tune.loguniform(lo, hi)` | Log-scale sampling — always use for LR, weight decay, regularisation |
| `tune.choice([...])` | Uniform random pick from a list of discrete values |
| `Tuner.fit()` → `ResultGrid` | Blocks until all trials finish; returns `.get_best_result()` and `.get_dataframe()` |
| `ASHAScheduler` | Early stopping: survivors grow; bottom fraction stopped at each halving round |
| `OptunaSearch` | Bayesian TPE sampler — proposes smarter configs from past trial results |
| ASHA + Optuna | **Best default combination** — smart proposals + early stopping |
| `PopulationBasedTraining` | Mutates hyperparameters during training; needs checkpointing; best for long RL runs |

> **Tip:** ASHA (Asynchronous Successive Halving) is the right default scheduler for most HPO jobs. Start with ASHA + Optuna as your baseline combination.

---
## What's next
**Day 7** → Ray Serve — Deploying Models as Scalable REST APIs. You'll define deployments with `@serve.deployment`, build a model serving pipeline, and scale replicas with a YAML config.

Mark Day 6 complete in your [tracker](../index.html).
